# Bulk clinical associations

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Figure 1 — Analysis

This notebook reads only the paper-wide public-source manifest produced by
`00_Paper_Data_Download.ipynb` and writes panel-ready tables.

It performs no downloads and creates no figures.

Outputs are written to `data/processed/figure_1_clean/`:

- `panel_b_grade_data.csv` and `panel_b_grade_summary.csv`
- `panel_c_stage_data.csv` and `panel_c_stage_summary.csv`
- `panel_d_cptac_coherence.csv`
- `panel_e_tracerx_resamples.csv` and `panel_e_tracerx_summary.csv`
- `table_1_emtab_adjusted_models.csv`
- `figure_1_processed_manifest.csv`


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import spearmanr

PROJECT_ROOT = Path(os.environ["MITOXY_PROJECT_ROOT"])
RAW_MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" / "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_1_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3 = ["BAK1", "BAX", "BID"]
RANDOM_SEED = 20260913

raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)


def source_path(dataset, role):
    matches = raw_manifest.loc[
        raw_manifest["dataset"].eq(dataset)
        & raw_manifest["role"].eq(role)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one public source for ({dataset!r}, {role!r}); "
            f"found {len(matches)}"
        )
    row = matches.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    return path


RAW = {
    "tcga_xena_expression": source_path(
        "TCGA-KIRC", "tumor RNA expression"
    ),
    "tcga_gdc_clinical_json": source_path(
        "TCGA-KIRC", "clinical API response"
    ),
    "cptac_rna": source_path(
        "CPTAC ccRCC", "tumor RNA expression"
    ),
    "cptac_protein": source_path(
        "CPTAC ccRCC", "tumor protein abundance"
    ),
    "cptac_clinical_xlsx": source_path(
        "CPTAC ccRCC", "clinical supplement"
    ),
    "emtab_expression": source_path(
        "E-MTAB-1980", "ccRCC_exp_log_quantile_normalized.txt"
    ),
    "emtab_clinical_xlsx": source_path(
        "E-MTAB-1980",
        "Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx",
    ),
    "tracerx_tpm": source_path(
        "TRACERx Renal", "regional tumor TPM"
    ),
    "tracerx_annotation": source_path(
        "TRACERx Renal", "regional annotation"
    ),
    "tracerx_clinical": source_path(
        "TRACERx Renal", "clinical annotation"
    ),
}

print("Paper-wide public-source manifest loaded:", RAW_MANIFEST_PATH)
display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))


In [ ]:
def zscore(series):
    values = pd.to_numeric(series, errors="coerce")
    return (values - values.mean()) / values.std(ddof=0)


def equal4_score(frame):
    standardized = frame[GENES].apply(zscore, axis=0)
    return standardized.mean(axis=1)


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}[match.group(1)]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def parse_t_category(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"T([1-4])", str(value).upper())
    return float(match.group(1)) if match else np.nan


def parse_m1(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper()
    if re.search(r"M1", text):
        return 1.0
    if re.search(r"M0", text):
        return 0.0
    return np.nan


def correlation_row(cohort, endpoint, data, outcome_column):
    complete = data[["mitoxy", outcome_column]].dropna()
    rho, p_value = spearmanr(complete["mitoxy"], complete[outcome_column])
    z_value = np.arctanh(np.clip(rho, -0.999999, 0.999999))
    se = 1 / np.sqrt(len(complete) - 3)
    return {
        "cohort": cohort,
        "endpoint": endpoint,
        "n": len(complete),
        "rho": float(rho),
        "ci_low": float(np.tanh(z_value - 1.95996398454 * se)),
        "ci_high": float(np.tanh(z_value + 1.95996398454 * se)),
        "p_value": float(p_value),
    }


def read_linkedomics(path):
    matrix = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    matrix = matrix.apply(pd.to_numeric, errors="coerce")
    matrix.index = [str(value).split("|")[0].strip().upper() for value in matrix.index]
    matrix.columns = [str(value).strip().upper() for value in matrix.columns]
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()
    return matrix


def find_excel_header(path, required_terms, maximum_rows=20):
    preview = pd.read_excel(path, header=None, nrows=maximum_rows)
    best_row, best_score = None, -1
    for row_number, row in preview.iterrows():
        text = " | ".join(row.dropna().astype(str)).lower()
        score = sum(term.lower() in text for term in required_terms)
        if score > best_score:
            best_row, best_score = row_number, score
    return int(best_row)


def column_containing(columns, terms, required=True):
    for column in columns:
        normalized = str(column).strip().lower()
        if any(term.lower() in normalized for term in terms):
            return column
    if required:
        raise KeyError(f"No column matched {terms}; available: {list(columns)}")
    return None


## TCGA-KIRC


In [ ]:
from mitoxy.clinical import primary_diagnoses
def first_numeric(records,key,parser):
    values=[parser(record.get(key)) for record in records]
    values=[x for x in values if pd.notna(x)]
    if len(set(values))>1: raise ValueError(f"Conflicting GDC values for {key}: {values}")
    return values[0] if values else np.nan

tcga_expression = pd.read_csv(
    RAW["tcga_xena_expression"], sep="\t", index_col=0
)
tcga_expression.index = tcga_expression.index.astype(str).str.upper()

missing = sorted(set(GENES) - set(tcga_expression.index))
assert not missing, f"TCGA expression missing {missing}"

tcga_gene = tcga_expression.loc[GENES].T.apply(pd.to_numeric, errors="coerce")
tcga_gene.index = tcga_gene.index.astype(str).str.upper()

# Primary-tumor samples only. Xena barcodes commonly end in ``-01``;
# some releases append an aliquot letter such as ``-01A``.
primary_mask = tcga_gene.index.str.match(
    r"^TCGA-[A-Z0-9]+-[A-Z0-9]+-01",
    na=False,
)
tcga_gene = tcga_gene.loc[primary_mask].copy()
tcga_gene["sample_id"] = tcga_gene.index.str.slice(0, 12)
tcga_gene = tcga_gene.groupby("sample_id")[GENES].mean()
assert len(tcga_gene) == 533, (
    f"Expected 533 TCGA primary-tumor cases; found {len(tcga_gene)}"
)
tcga_gene["mitoxy"] = equal4_score(tcga_gene)

clinical_object = json.loads(RAW["tcga_gdc_clinical_json"].read_text())
if isinstance(clinical_object, list):
    cases = clinical_object
elif "data" in clinical_object:
    cases = clinical_object["data"].get("hits", clinical_object["data"])
else:
    cases = clinical_object.get("hits", clinical_object)

clinical_rows = []
for case in cases:
    if not isinstance(case, dict):
        continue
    case_id = case.get("submitter_id")
    diagnoses = primary_diagnoses(case)
    diagnosis = diagnoses[0] if diagnoses else {}
    clinical_rows.append({
        "sample_id": case_id,
        "grade": first_numeric(diagnoses,"tumor_grade",parse_grade),
        "stage": first_numeric(diagnoses,"ajcc_pathologic_stage",parse_stage),
    })

tcga_clinical = pd.DataFrame(clinical_rows).drop_duplicates("sample_id")
tcga = tcga_gene.reset_index().merge(tcga_clinical, on="sample_id", how="left")
tcga.insert(0, "cohort", "TCGA-KIRC")

print("TCGA cases:", len(tcga))
print(tcga[["grade", "stage"]].notna().sum())


## CPTAC ccRCC


In [ ]:
cptac_rna = read_linkedomics(RAW["cptac_rna"])
cptac_protein = read_linkedomics(RAW["cptac_protein"])

for matrix_name, matrix in [("RNA", cptac_rna), ("protein", cptac_protein)]:
    missing = sorted(set(GENES) - set(matrix.index))
    assert not missing, f"CPTAC {matrix_name} missing {missing}"

cptac_gene = cptac_rna.loc[GENES].T
cptac_gene.index.name = "sample_id"
cptac_gene["mitoxy"] = equal4_score(cptac_gene)

cptac_clinical_raw = pd.read_excel(
    RAW["cptac_clinical_xlsx"],
    sheet_name="ccrcc_clinical_characteristics",
    header=0,
    dtype=str,
)
cptac_clinical_raw.columns = [str(column).strip() for column in cptac_clinical_raw.columns]

# Table S1 contains seven non-clear-cell RCC cases. Figure 1 uses only the
# 103 histologically confirmed clear-cell RCC tumors.
histology_column = column_containing(
    cptac_clinical_raw.columns,
    ["histologic_type", "histologic type"],
)
cptac_clinical_raw = cptac_clinical_raw.loc[
    cptac_clinical_raw[histology_column]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("clear cell renal cell carcinoma")
].copy()
assert len(cptac_clinical_raw) == 103, (
    "Expected 103 clear-cell RCC cases in CPTAC Table S1; "
    f"found {len(cptac_clinical_raw)}"
)

# Identify the case-ID column by overlap with expression sample IDs.
sample_ids = set(cptac_gene.index.astype(str))
overlap_counts = {}
for column in cptac_clinical_raw.columns:
    values = set(cptac_clinical_raw[column].dropna().astype(str).str.strip())
    overlap_counts[column] = len(values & sample_ids)
id_column = max(overlap_counts, key=overlap_counts.get)
assert overlap_counts[id_column] >= 100

grade_column = column_containing(cptac_clinical_raw.columns, ["grade"])
stage_column = column_containing(cptac_clinical_raw.columns, ["stage"])

cptac_clinical = pd.DataFrame({
    "sample_id": cptac_clinical_raw[id_column].astype(str).str.strip(),
    "grade": cptac_clinical_raw[grade_column].map(parse_grade),
    "stage": cptac_clinical_raw[stage_column].map(parse_stage),
}).drop_duplicates("sample_id")

cptac = cptac_gene.reset_index().merge(cptac_clinical, on="sample_id", how="inner")
cptac.insert(0, "cohort", "CPTAC")

print("CPTAC cases:", len(cptac))
print(cptac[["grade", "stage"]].notna().sum())


## E-MTAB-1980


In [ ]:
emtab_expression = pd.read_csv(
    RAW["emtab_expression"], sep="\t", index_col=0, low_memory=False
)

probe_map = {
    "BAK1": ["A_23_P145357"],
    # The archived analysis selected the RefSeq-mapped BAX probe below;
    # A_23_P208706 was not part of the frozen score definition.
    "BAX": ["A_33_P3341144"],
    "BID": ["A_24_P187948", "A_33_P3227041"],
    "SLC7A11": ["A_33_P3242623"],
}

biological_columns = [
    column for column in emtab_expression.columns
    if re.fullmatch(r"ccRCC-\d+", str(column), flags=re.I)
]
assert len(biological_columns) == 101

emtab_gene = pd.DataFrame(index=biological_columns)
for gene, probes in probe_map.items():
    missing_probes = sorted(set(probes) - set(emtab_expression.index.astype(str)))
    assert not missing_probes, f"Missing {gene} probes: {missing_probes}"
    emtab_gene[gene] = (
        emtab_expression.loc[probes, biological_columns]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=0)
    )

emtab_gene.index.name = "sample_id"
emtab_gene["mitoxy"] = equal4_score(emtab_gene)

emtab_clinical_raw = pd.read_excel(
    RAW["emtab_clinical_xlsx"], header=1, dtype=str
)
emtab_clinical_raw.columns = [str(column).strip() for column in emtab_clinical_raw.columns]

emtab_clinical = pd.DataFrame({
    "sample_id": emtab_clinical_raw["sample ID"].astype(str).str.strip(),
    "sex_raw": emtab_clinical_raw["Sex"],
    "age": pd.to_numeric(emtab_clinical_raw["Age"], errors="coerce"),
    "grade": emtab_clinical_raw["Fuhrman grade"].map(parse_grade),
    "stage_raw": emtab_clinical_raw["Stage at diagnosis"],
})
emtab_clinical["stage"] = emtab_clinical["stage_raw"].map(parse_t_category)
emtab_clinical["m1"] = emtab_clinical["stage_raw"].map(parse_m1)
emtab_clinical["male"] = emtab_clinical["sex_raw"].astype(str).str.upper().eq("M").astype(float)
emtab_clinical["t34"] = np.where(
    emtab_clinical["stage"].notna(),
    (emtab_clinical["stage"] >= 3).astype(float),
    np.nan,
)

emtab = emtab_gene.reset_index().merge(emtab_clinical, on="sample_id", how="left")
emtab.insert(0, "cohort", "E-MTAB-1980")

print("E-MTAB cases:", len(emtab))
print(emtab[["grade", "stage"]].notna().sum())


## TRACERx Renal


In [ ]:
rds = pyreadr.read_r(str(RAW["tracerx_tpm"]))
tracerx_expression = rds[None]
tracerx_expression.index = tracerx_expression.index.astype(str).str.upper()

annotation = pd.read_csv(RAW["tracerx_annotation"], sep="\t", dtype=str)
annotation.columns = [str(column).strip() for column in annotation.columns]

sample_column = column_containing(annotation.columns, ["sample"])
patient_column = column_containing(annotation.columns, ["patient"])
type_column = column_containing(annotation.columns, ["type_collapsed", "type collapsed"])

primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.upper().eq("PRIMARY")
].copy()

primary_samples = [
    sample for sample in primary_annotation[sample_column].astype(str)
    if sample in tracerx_expression.columns
]
assert len(primary_samples) == 191

region_gene = (
    tracerx_expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
region_gene = np.log2(region_gene + 1)
region_gene["mitoxy"] = equal4_score(region_gene)
region_gene["sample_id"] = region_gene.index.astype(str)

region_gene = region_gene.merge(
    primary_annotation[[sample_column, patient_column]].rename(
        columns={sample_column: "sample_id", patient_column: "patient"}
    ),
    on="sample_id",
    how="left",
    validate="one_to_one",
)

clinical = pd.read_csv(RAW["tracerx_clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(column).strip() for column in clinical.columns]

tracerx_clinical = pd.DataFrame({
    "patient": clinical["Subject"].astype(str).str.strip(),
    "grade": clinical["Fuhrman Grade"].map(parse_grade),
    "stage": clinical["Overall Stage"].map(parse_stage),
})

tracerx_patient = (
    region_gene.groupby("patient", as_index=False)["mitoxy"].mean()
    .merge(tracerx_clinical, on="patient", how="left")
)
tracerx_patient = tracerx_patient.rename(columns={"patient": "sample_id"})
tracerx_patient.insert(0, "cohort", "TRACERx")

# Preserve the archived direct-ID merge behavior: 73 matched clinical patients.
tracerx_patient = tracerx_patient.dropna(subset=["grade", "stage"]).copy()
assert len(tracerx_patient) == 73

print("TRACERx primary regions:", len(region_gene))
print("TRACERx clinical patients:", len(tracerx_patient))


## Panels B and C — cohort-level associations


In [ ]:
cohort_frames = {
    "TCGA-KIRC": tcga,
    "CPTAC": cptac,
    "E-MTAB-1980": emtab,
    "TRACERx": tracerx_patient,
}

panel_b_data = pd.concat([
    frame[["cohort", "sample_id", "mitoxy", "grade"]]
    for frame in cohort_frames.values()
], ignore_index=True)

panel_c_data = pd.concat([
    frame[["cohort", "sample_id", "mitoxy", "stage"]]
    for frame in cohort_frames.values()
], ignore_index=True)

panel_b_summary = pd.DataFrame([
    correlation_row(cohort, "Grade", frame, "grade")
    for cohort, frame in cohort_frames.items()
])

panel_c_summary = pd.DataFrame([
    correlation_row(
        cohort,
        "T category" if cohort == "E-MTAB-1980" else "Stage",
        frame,
        "stage",
    )
    for cohort, frame in cohort_frames.items()
])

display(panel_b_summary)
display(panel_c_summary)






panel_b_data.to_csv(OUTPUT_DIR / "panel_b_grade_data.csv", index=False)
panel_b_summary.to_csv(OUTPUT_DIR / "panel_b_grade_summary.csv", index=False)
panel_c_data.to_csv(OUTPUT_DIR / "panel_c_stage_data.csv", index=False)
panel_c_summary.to_csv(OUTPUT_DIR / "panel_c_stage_summary.csv", index=False)


## Revised panel D — CPTAC cross-molecular coherence


In [ ]:
cptac_ids = (
    cptac.loc[cptac["grade"].notna(), "sample_id"]
    .drop_duplicates()
    .tolist()
)
assert len(cptac_ids) == 103

cptac_multi = pd.DataFrame(index=cptac_ids)
for gene in GENES:
    cptac_multi[f"{gene}_rna"] = cptac_rna.loc[gene, cptac_ids]
    cptac_multi[f"{gene}_protein"] = cptac_protein.loc[gene, cptac_ids]
    cptac_multi[f"{gene}_rna_z"] = zscore(cptac_multi[f"{gene}_rna"])
    cptac_multi[f"{gene}_protein_z"] = zscore(cptac_multi[f"{gene}_protein"])

cptac_multi["rna_core3"] = cptac_multi[[f"{g}_rna_z" for g in CORE3]].mean(axis=1)
cptac_multi["protein_core3"] = cptac_multi[[f"{g}_protein_z" for g in CORE3]].mean(axis=1)

comparisons = [
    ("Core3 RNA vs protein", "rna_core3", "protein_core3"),
    ("SLC7A11 RNA vs BID RNA", "SLC7A11_rna", "BID_rna"),
    ("SLC7A11 RNA vs BID protein", "SLC7A11_rna", "BID_protein"),
]

rows = []
for label, x_column, y_column in comparisons:
    complete = cptac_multi[[x_column, y_column]].dropna()
    rho, p_value = spearmanr(complete[x_column], complete[y_column])
    z_value = np.arctanh(np.clip(rho, -0.999999, 0.999999))
    se = 1 / np.sqrt(len(complete) - 3)
    rows.append({
        "comparison": label,
        "n": len(complete),
        "rho": float(rho),
        "ci_low": float(np.tanh(z_value - 1.95996398454 * se)),
        "ci_high": float(np.tanh(z_value + 1.95996398454 * se)),
        "p_value": float(p_value),
    })

panel_d = pd.DataFrame(rows)
display(panel_d)

panel_d.to_csv(OUTPUT_DIR / "panel_d_cptac_coherence.csv", index=False)


## Revised panel E — TRACERx one-region sampling


In [ ]:
eligible_patients = set(tracerx_patient["sample_id"])
resampling_regions = region_gene.loc[region_gene["patient"].isin(eligible_patients)].copy()
resampling_regions = resampling_regions.merge(
    tracerx_clinical, on="patient", how="left", validate="many_to_one"
)
assert resampling_regions["patient"].nunique() == 73

patient_groups = {
    patient: group.reset_index(drop=True)
    for patient, group in resampling_regions.groupby("patient", sort=True)
}

rng = np.random.default_rng(RANDOM_SEED)
resample_rows = []
for iteration in range(1000):
    selected = []
    for patient in sorted(patient_groups):
        group = patient_groups[patient]
        selected.append(group.iloc[rng.integers(0, len(group))])
    sample = pd.DataFrame(selected)
    rho_grade, p_grade = spearmanr(sample["mitoxy"], sample["grade"])
    rho_stage, p_stage = spearmanr(sample["mitoxy"], sample["stage"])
    resample_rows.append({
        "iteration": iteration + 1,
        "rho_grade": float(rho_grade),
        "p_grade": float(p_grade),
        "rho_stage": float(rho_stage),
        "p_stage": float(p_stage),
        "n": len(sample),
    })

panel_e_resamples = pd.DataFrame(resample_rows)
panel_e_summary = pd.DataFrame([
    {
        "endpoint": endpoint,
        "n_repeats": len(panel_e_resamples),
        "median": panel_e_resamples[column].median(),
        "q025": panel_e_resamples[column].quantile(0.025),
        "q975": panel_e_resamples[column].quantile(0.975),
        "direction_positive_frac": (panel_e_resamples[column] > 0).mean(),
    }
    for endpoint, column in [("Grade", "rho_grade"), ("Stage", "rho_stage")]
])

display(panel_e_summary)

panel_e_resamples.to_csv(OUTPUT_DIR / "panel_e_tracerx_resamples.csv", index=False)
panel_e_summary.to_csv(OUTPUT_DIR / "panel_e_tracerx_summary.csv", index=False)


## Table 1 candidate — adjusted E-MTAB-1980 models


In [ ]:
emtab_models = emtab.copy()
emtab_models["mitoxy_sd"] = zscore(emtab_models["mitoxy"])


def adjusted_linear(outcome):
    data = emtab_models[[outcome, "mitoxy_sd", "age", "male"]].dropna()
    X = sm.add_constant(data[["mitoxy_sd", "age", "male"]], has_constant="add")
    model = sm.OLS(data[outcome], X).fit(cov_type="HC3", use_t=False)
    ci = model.conf_int().loc["mitoxy_sd"]
    return {
        "endpoint": outcome,
        "effect_type": "beta",
        "n": len(data),
        "effect": model.params["mitoxy_sd"],
        "ci_low": ci.iloc[0],
        "ci_high": ci.iloc[1],
        "p_value": model.pvalues["mitoxy_sd"],
    }


def adjusted_logistic(outcome):
    data = emtab_models[[outcome, "mitoxy_sd", "age", "male"]].dropna()
    X = sm.add_constant(data[["mitoxy_sd", "age", "male"]], has_constant="add")
    model = sm.Logit(data[outcome], X).fit(disp=False)
    ci = model.conf_int().loc["mitoxy_sd"]
    return {
        "endpoint": outcome,
        "effect_type": "odds_ratio",
        "n": len(data),
        "effect": np.exp(model.params["mitoxy_sd"]),
        "ci_low": np.exp(ci.iloc[0]),
        "ci_high": np.exp(ci.iloc[1]),
        "p_value": model.pvalues["mitoxy_sd"],
    }


table_1 = pd.DataFrame([
    adjusted_linear("grade"),
    adjusted_linear("stage"),
    adjusted_logistic("t34"),
    adjusted_logistic("m1"),
])
table_1["endpoint"] = ["Fuhrman grade", "T category", "T3–T4 disease", "M1 disease"]

display(table_1)

table_1.to_csv(OUTPUT_DIR / "table_1_emtab_adjusted_models.csv", index=False)


## Processed-file contract


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


PROCESSED_MANIFEST_PATH = OUTPUT_DIR / "figure_1_processed_manifest.csv"
processed_files = sorted(
    path
    for path in OUTPUT_DIR.glob("*.csv")
    if path != PROCESSED_MANIFEST_PATH
)
processed_manifest = pd.DataFrame([
    {
        "file": path.name,
        "path": str(path.resolve()),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in processed_files
])

processed_manifest.to_csv(PROCESSED_MANIFEST_PATH, index=False)

display(processed_manifest)
print("Figure 1 analysis handoff: PASS")
print(PROCESSED_MANIFEST_PATH)
